# Legal-Harness C5 speed–quality diagnostic

## tl;dr

This notebook reconciles the C5 full-226 wall clock, D4/C5 per-question evaluator latency, ParSeR reproduced timing, and every recorded C5 skill attempt. It separates observed facts from counterfactual speed targets. The decision question is whether C5 can preserve its precision, stability, and citation guarantees while recovering D4's 2-hop complete-evidence rate and approaching ParSeR's throughput.

## Context & Methods

### Key assumptions

- C5 and D4 quality metrics use the same 226 KoBLEX questions and the repository evaluator.
- C5 batch wall time is the sum of the six immutable chunk summaries; this reconciles to the published 31,353.105 seconds.
- D4 latency is the recorded per-question inner latency because a single clean batch wall clock is unavailable across its mixed-provenance run.
- ParSeR has 226 attempted items, 220 scored results, six execution errors, and no ABSTAIN policy. Its 7,924.5-second batch wall clock is the fair throughput comparator; its 30.231-second mean is successful-generation latency.
- Skill-attempt timestamps record completions, not model start times. Inter-attempt intervals include any retrieval/control work preceding that model completion.
- A local 2026-08-24 microbenchmark measured repeated resource initialization at 8.748 seconds and warm KURE+BGE retrieval at 0.268 seconds after first-call warm-up. These are measurements of the current machine, not full-run counterfactuals.

## Data

In [1]:
from pathlib import Path
from datetime import datetime
import csv
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "scripts/evaluate_dev_runs.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "scripts/evaluate_dev_runs.py").is_file(), ROOT
C5_EVAL = ROOT / "records/evaluations/kure-c5-full226-20260822-a74e76d-checkpoint-226/per_question.csv"
D4_EVAL = ROOT / "records/evaluations/kure-bge-abstain-repair-d4-qwen38-q8-226-20260820-reconciled-20260821/per_question.csv"
C5_AGG = ROOT / "records/evaluations/kure-c5-full226-20260822-a74e76d-checkpoint-226/aggregate.json"
D4_AGG = ROOT / "records/evaluations/kure-bge-abstain-repair-d4-qwen38-q8-226-20260820-reconciled-20260821/aggregate.json"
PARSER_AGG = ROOT / "records/evaluations/koblex-parser-reproduced-qwen38-q8-226-20260819/aggregate.json"
PARSER_RUN_SUMMARY = ROOT / "records/baselines/koblex-parser-reproduced-qwen38-q8-226-23f5c9ae-45a5-4d7e-a88f-fbcfb5320185/run_summary.json"
PARSER_RESULTS = ROOT / "records/baselines/koblex-parser-reproduced-qwen38-q8-226-23f5c9ae-45a5-4d7e-a88f-fbcfb5320185/final_results.jsonl"
C5_RUN_ROOT = ROOT / "records/runs/kure-c5-full226-20260822-a74e76d"
C5_BATCH_ROOT = ROOT / "records/batches/kure-c5-full226-20260822-a74e76d"

RESOURCE_INIT_SECONDS = 8.748
WARM_RETRIEVAL_FIRST_SECONDS = 0.687
WARM_RETRIEVAL_STEADY_SECONDS = 0.268
C5_PUBLISHED_BATCH_WALL_SECONDS = 31353.105

for required in [C5_EVAL, D4_EVAL, C5_AGG, D4_AGG, PARSER_AGG, PARSER_RUN_SUMMARY, PARSER_RESULTS, C5_RUN_ROOT, C5_BATCH_ROOT]:
    assert required.exists(), required

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def parse_time(value):
    return datetime.fromisoformat(value)

c5 = pd.read_csv(C5_EVAL)
d4 = pd.read_csv(D4_EVAL)
c5_agg = read_json(C5_AGG)
d4_agg = read_json(D4_AGG)
parser_agg = read_json(PARSER_AGG)
parser_run_summary = read_json(PARSER_RUN_SUMMARY)
parser_results = pd.DataFrame(read_jsonl(PARSER_RESULTS))

chunk_dirs = sorted(C5_BATCH_ROOT.glob("kure-c5-full226-20260822-a74e76d-chunk-*"))
outer_rows = []
for chunk_dir in chunk_dirs:
    outer_rows.extend(read_jsonl(chunk_dir / "summary.jsonl"))
outer = pd.DataFrame(outer_rows)
outer["outer_wall_s"] = (
    pd.to_datetime(outer["completed_at_utc"], utc=True)
    - pd.to_datetime(outer["started_at_utc"], utc=True)
).dt.total_seconds()

print({
    "c5_rows": len(c5),
    "d4_rows": len(d4),
    "parser_rows": len(parser_results),
    "c5_chunk_count": len(chunk_dirs),
    "c5_outer_rows": len(outer),
})

{'c5_rows': 226, 'd4_rows': 226, 'parser_rows': 226, 'c5_chunk_count': 6, 'c5_outer_rows': 226}


### Reconstruct every C5 model attempt and its elapsed interval

In [2]:
attempt_rows = []
run_rows = []

for row in c5.itertuples(index=False):
    run_dir = C5_RUN_ROOT / str(row.run_id)
    metadata = read_json(run_dir / "metadata.json")
    result = read_json(run_dir / "result.json")
    attempts = read_jsonl(run_dir / "skill_attempts.jsonl") if (run_dir / "skill_attempts.jsonl").exists() else []
    previous_at = parse_time(metadata["started_at_utc"])
    invocation_counts = {}
    total_tokens = 0

    for sequence, attempt in enumerate(attempts, start=1):
        recorded_at = parse_time(attempt["recorded_at_utc"])
        interval_s = (recorded_at - previous_at).total_seconds()
        previous_at = recorded_at
        entry_point = str(attempt["entry_point"])
        if int(attempt.get("attempt", 1)) == 1:
            invocation_counts[entry_point] = invocation_counts.get(entry_point, 0) + 1
        invocation_index = invocation_counts.get(entry_point, 1)
        response_tokens = int(attempt.get("response_tokens") or attempt.get("eval_count") or 0)
        total_tokens += response_tokens
        attempt_rows.append({
            "question_id": row.question_id,
            "run_id": row.run_id,
            "sequence": sequence,
            "skill_id": attempt["skill_id"],
            "entry_point": entry_point,
            "invocation_index": invocation_index,
            "attempt": int(attempt.get("attempt", 1)),
            "outcome": attempt.get("outcome"),
            "done_reason": attempt.get("done_reason"),
            "response_tokens": response_tokens,
            "response_characters": int(attempt.get("response_characters") or 0),
            "requested_max_tokens": int(attempt.get("requested_max_tokens") or 0),
            "interval_s": interval_s,
        })

    completed_at = parse_time(result["completed_at_utc"])
    tail_s = (completed_at - previous_at).total_seconds() if attempts else (completed_at - parse_time(metadata["started_at_utc"])).total_seconds()
    run_rows.append({
        "question_id": row.question_id,
        "run_id": row.run_id,
        "attempt_count": len(attempts),
        "total_response_tokens": total_tokens,
        "retry_or_invalid_attempts": sum(1 for item in attempts if int(item.get("attempt", 1)) > 1 or item.get("outcome") != "valid"),
        "s1_attempts": sum(item.get("skill_id") == "S1" for item in attempts),
        "s2_attempts": sum(item.get("skill_id") == "S2" for item in attempts),
        "s3_attempts": sum(item.get("skill_id") == "S3" for item in attempts),
        "tail_s": tail_s,
    })

attempts = pd.DataFrame(attempt_rows)
run_features = pd.DataFrame(run_rows)
c5["latency_s"] = c5["end_to_end_latency_ms"] / 1000.0
d4["latency_s"] = d4["end_to_end_latency_ms"] / 1000.0
c5_detail = (
    c5.merge(run_features, on=["question_id", "run_id"], how="left", validate="one_to_one")
      .merge(outer[["question_id", "outer_wall_s"]], on="question_id", how="left", validate="one_to_one")
)
c5_detail["outer_minus_inner_s"] = c5_detail["outer_wall_s"] - c5_detail["latency_s"]

assert len(c5_detail) == 226
assert c5_detail["run_id"].nunique() == 226
assert len(attempts) == int(c5_detail["attempt_count"].sum())
assert c5_detail[["outer_wall_s", "attempt_count", "total_response_tokens"]].notna().all().all()

print({
    "skill_attempts": len(attempts),
    "generated_tokens": int(attempts["response_tokens"].sum()),
    "retry_or_invalid_attempts": int(c5_detail["retry_or_invalid_attempts"].sum()),
})

{'skill_attempts': 1126, 'generated_tokens': 1131133, 'retry_or_invalid_attempts': 92}


### Reconcile source totals before interpreting drivers

In [3]:
checks = pd.DataFrame([
    {
        "check": "C5 chunk wall sum",
        "observed": c5_detail["outer_wall_s"].sum(),
        "expected": 31353.105,
        "absolute_difference": abs(c5_detail["outer_wall_s"].sum() - 31353.105),
    },
    {
        "check": "C5 evaluator latency sum",
        "observed": c5_detail["latency_s"].sum(),
        "expected": 30810.179412,
        "absolute_difference": abs(c5_detail["latency_s"].sum() - 30810.179412),
    },
    {
        "check": "ParSeR batch wall",
        "observed": float(parser_run_summary["wall_seconds"]),
        "expected": 7924.5,
        "absolute_difference": abs(float(parser_run_summary["wall_seconds"]) - 7924.5),
    },
])
assert checks.loc[checks["check"] != "C5 chunk wall sum", "absolute_difference"].max() < 0.01
assert checks.loc[checks["check"] == "C5 chunk wall sum", "absolute_difference"].iloc[0] / C5_PUBLISHED_BATCH_WALL_SECONDS < 0.002
display(checks.round(6))

,check,observed,expected,absolute_difference
0,C5 chunk wall sum,31322.158453,31353.105000,30.946547
1,C5 evaluator latency sum,30810.179412,30810.179412,0.000000
2,ParSeR batch wall,7924.500000,7924.500000,0.000000


## Results

### C5 is much faster than D4, but remains about four times slower than ParSeR

In [4]:
parser_success = parser_results[parser_results["error"].fillna("").eq("")]
comparison = pd.DataFrame([
    {
        "system": "KURE D4",
        "attempted_items": len(d4),
        "scored_or_normal_items": int((d4["status"] != "EXECUTION_FAILURE").sum()),
        "answer_rate": float(d4_agg["answer_quality"]["answer_rate"]),
        "execution_failure_rate": float(d4_agg["answer_quality"]["execution_failure_rate"]),
        "provision_macro_f1": float(d4_agg["provision"]["macro"]["f1"]),
        "provision_macro_recall": float(d4_agg["provision"]["macro"]["recall"]),
        "complete_evidence_rate": float(d4_agg["provision"]["complete_evidence"]["rate"]),
        "token_f1_e2e": float(d4_agg["answer_quality"]["token_f1_at_800"]["end_to_end"]),
        "mean_recorded_latency_s": float(d4["latency_s"].mean()),
        "batch_wall_s": np.nan,
    },
    {
        "system": "KURE C5",
        "attempted_items": len(c5_detail),
        "scored_or_normal_items": int((c5_detail["status"] != "EXECUTION_FAILURE").sum()),
        "answer_rate": float(c5_agg["answer_quality"]["answer_rate"]),
        "execution_failure_rate": float(c5_agg["answer_quality"]["execution_failure_rate"]),
        "provision_macro_f1": float(c5_agg["provision"]["macro"]["f1"]),
        "provision_macro_recall": float(c5_agg["provision"]["macro"]["recall"]),
        "complete_evidence_rate": float(c5_agg["provision"]["complete_evidence"]["rate"]),
        "token_f1_e2e": float(c5_agg["answer_quality"]["token_f1_at_800"]["end_to_end"]),
        "mean_recorded_latency_s": float(c5_detail["latency_s"].mean()),
        "batch_wall_s": C5_PUBLISHED_BATCH_WALL_SECONDS,
    },
    {
        "system": "ParSeR Reproduced",
        "attempted_items": int(parser_agg["n_items"]),
        "scored_or_normal_items": int(parser_agg["n_scored"]),
        "answer_rate": float(parser_agg["answer_rate"]),
        "execution_failure_rate": float(parser_agg["n_errors"]) / float(parser_agg["n_items"]),
        "provision_macro_f1": float(parser_agg["retrieval"]["f1_mean"]) * float(parser_agg["n_scored"]) / float(parser_agg["n_items"]),
        "provision_macro_recall": float(parser_agg["retrieval"]["recall_mean"]) * float(parser_agg["n_scored"]) / float(parser_agg["n_items"]),
        "complete_evidence_rate": np.nan,
        "token_f1_e2e": float(parser_agg["token_f1"]["mean"]) * float(parser_agg["n_scored"]) / float(parser_agg["n_items"]),
        "mean_recorded_latency_s": float(parser_agg["latency_seconds"]["total_mean"]),
        "batch_wall_s": float(parser_run_summary["wall_seconds"]),
    },
])
comparison["batch_wall_hours"] = comparison["batch_wall_s"] / 3600.0
display(comparison.round(4))

c5_vs_d4_speedup = d4["latency_s"].mean() / c5_detail["latency_s"].mean()
parser_vs_c5_wall_speedup = (C5_PUBLISHED_BATCH_WALL_SECONDS / len(c5_detail)) / (float(parser_run_summary["wall_seconds"]) / int(parser_agg["n_items"]))
print({
    "C5_speedup_vs_D4_recorded_mean": round(c5_vs_d4_speedup, 3),
    "ParSeR_speedup_vs_C5_batch_wall_per_item": round(parser_vs_c5_wall_speedup, 3),
})

,system,attempted_items,scored_or_normal_items,answer_rate,execution_failure_rate,provision_macro_f1,provision_macro_recall,complete_evidence_rate,token_f1_e2e,mean_recorded_latency_s,batch_wall_s,batch_wall_hours
0,KURE D4,226,186,0.6549,0.1770,0.5844,0.6969,0.6062,0.1710,341.4478,NaN,NaN
1,KURE C5,226,222,0.8274,0.0177,0.6706,0.7220,0.5487,0.3019,136.3282,31353.105,8.7092
2,ParSeR Reproduced,226,220,0.9735,0.0265,0.5208,0.5101,NaN,0.4244,30.2310,7924.500,2.2012


{'C5_speedup_vs_D4_recorded_mean': 2.505, 'ParSeR_speedup_vs_C5_batch_wall_per_item': 3.956}


### Round count is the strongest operational segmentation of C5 latency

In [5]:
def q90(series):
    return series.quantile(0.90)

by_round = (
    c5_detail.groupby("retrieval_rounds", dropna=False)
    .agg(
        items=("question_id", "size"),
        mean_latency_s=("latency_s", "mean"),
        median_latency_s=("latency_s", "median"),
        p90_latency_s=("latency_s", q90),
        mean_attempts=("attempt_count", "mean"),
        mean_generated_tokens=("total_response_tokens", "mean"),
        mean_requests=("retrieval_requests", "mean"),
        answer_rate=("status", lambda s: (s == "ANSWER").mean()),
        complete_evidence_rate=("accepted_complete_evidence", "mean"),
        provision_f1=("provision_f1", "mean"),
    )
    .reset_index()
)
display(by_round.round(3))

by_hop = (
    c5_detail.groupby("n_hops")
    .agg(
        items=("question_id", "size"),
        mean_latency_s=("latency_s", "mean"),
        median_latency_s=("latency_s", "median"),
        mean_rounds=("retrieval_rounds", "mean"),
        mean_attempts=("attempt_count", "mean"),
        mean_generated_tokens=("total_response_tokens", "mean"),
        complete_evidence_rate=("accepted_complete_evidence", "mean"),
        provision_f1=("provision_f1", "mean"),
    )
    .reset_index()
)
display(by_hop.round(3))

,retrieval_rounds,items,mean_latency_s,median_latency_s,p90_latency_s,mean_attempts,mean_generated_tokens,mean_requests,answer_rate,complete_evidence_rate,provision_f1
0,0,2,61.158,61.158,74.747,2.000,2314.000,0.000,0.000,0.000,0.000
1,1,96,83.352,82.016,105.578,3.083,2976.479,1.740,0.969,0.740,0.808
2,2,52,141.252,135.483,168.006,5.327,5154.808,4.000,0.769,0.423,0.510
3,3,76,201.855,195.367,245.981,7.224,7535.697,6.105,0.711,0.408,0.625


,n_hops,items,mean_latency_s,median_latency_s,mean_rounds,mean_attempts,mean_generated_tokens,complete_evidence_rate,provision_f1
0,1,57,90.462,70.493,1.421,3.930,3068.018,0.825,0.751
1,2,127,142.146,139.847,2.016,5.236,5230.661,0.543,0.673
2,3,42,180.985,169.068,2.167,5.643,6951.476,0.190,0.556


### Model generations, not warm retrieval, dominate the observed path

In [6]:
attempt_summary = (
    attempts.groupby(["skill_id", "entry_point"])
    .agg(
        attempts=("run_id", "size"),
        unique_questions=("question_id", "nunique"),
        total_response_tokens=("response_tokens", "sum"),
        mean_response_tokens=("response_tokens", "mean"),
        median_interval_s=("interval_s", "median"),
        mean_interval_s=("interval_s", "mean"),
        total_interval_s=("interval_s", "sum"),
        invalid_or_retry=("outcome", lambda s: (s != "valid").sum()),
    )
    .reset_index()
)
attempt_summary["share_of_recorded_latency"] = attempt_summary["total_interval_s"] / c5_detail["latency_s"].sum()
display(attempt_summary.round(3))

s2_by_round = (
    attempts[attempts["entry_point"] == "ASSESS_COVERAGE"]
    .groupby("invocation_index")
    .agg(
        attempts=("run_id", "size"),
        questions=("question_id", "nunique"),
        mean_tokens=("response_tokens", "mean"),
        median_interval_s=("interval_s", "median"),
        mean_interval_s=("interval_s", "mean"),
    )
    .reset_index()
    .rename(columns={"invocation_index": "coverage_round"})
)
display(s2_by_round.round(3))

correlations = c5_detail[[
    "latency_s", "retrieval_rounds", "retrieval_requests", "attempt_count",
    "total_response_tokens", "candidate_count_unique"
]].corr(numeric_only=True)["latency_s"].sort_values(ascending=False)
display(correlations.to_frame("correlation_with_latency").round(3))

,skill_id,entry_point,attempts,unique_questions,total_response_tokens,mean_response_tokens,median_interval_s,mean_interval_s,total_interval_s,invalid_or_retry,share_of_recorded_latency
0,S1,GAP_QUERY_PLAN,204,128,113553,556.632,14.991,14.854,3030.283,0,0.098
1,S1,INITIAL_PLAN,251,226,299100,1191.633,36.512,37.366,9378.910,27,0.304
2,S2,ASSESS_COVERAGE,437,224,508201,1162.931,29.515,30.449,13306.264,11,0.432
3,S3,GENERATE_ANSWER,188,187,179182,953.096,20.291,21.674,4074.727,1,0.132
4,S3,GENERATE_BENCHMARK_CANDIDATE,46,35,31097,676.022,21.914,19.738,907.926,20,0.029


,coverage_round,attempts,questions,mean_tokens,median_interval_s,mean_interval_s
0,1,230,224,1080.361,27.207,27.600
1,2,131,128,1227.290,30.703,32.565
2,3,76,76,1301.882,33.023,35.425


,correlation_with_latency
latency_s,1.000
total_response_tokens,0.993
retrieval_requests,0.928
attempt_count,0.879
retrieval_rounds,0.852
candidate_count_unique,0.808


### The current process lifecycle adds a verified but non-dominant fixed cost

In [7]:
fixed_cost = pd.DataFrame([
    {"component": "Outer subprocess/check overhead", "seconds_per_item": c5_detail["outer_minus_inner_s"].mean(), "evidence": "C5 chunk wall minus evaluator latency"},
    {"component": "Repeated corpus + KURE + BGE initialization", "seconds_per_item": RESOURCE_INIT_SECONDS, "evidence": "Local H200 microbenchmark, 2026-08-24"},
    {"component": "Warm KURE+BGE request after warm-up", "seconds_per_item": WARM_RETRIEVAL_STEADY_SECONDS, "evidence": "One Top-100 KURE + BGE@100 request"},
])
display(fixed_cost.round(3))

observed_wall_mean = C5_PUBLISHED_BATCH_WALL_SECONDS / len(c5_detail)
persistent_warm_mean = observed_wall_mean - c5_detail["outer_minus_inner_s"].mean() - RESOURCE_INIT_SECONDS
print({
    "observed_C5_wall_mean_s": round(observed_wall_mean, 3),
    "persistent_warm_estimated_mean_s": round(persistent_warm_mean, 3),
    "estimated_reduction_percent": round(100 * (observed_wall_mean - persistent_warm_mean) / observed_wall_mean, 1),
    "estimated_full226_hours": round(persistent_warm_mean * 226 / 3600, 3),
})

,component,seconds_per_item,evidence
0,Outer subprocess/check overhead,2.265,C5 chunk wall minus evaluator latency
1,Repeated corpus + KURE + BGE initialization,8.748,"Local H200 microbenchmark, 2026-08-24"
2,Warm KURE+BGE request after warm-up,0.268,One Top-100 KURE + BGE@100 request


{'observed_C5_wall_mean_s': 138.731, 'persistent_warm_estimated_mean_s': 127.717, 'estimated_reduction_percent': 7.9, 'estimated_full226_hours': 8.018}


### Reaching ParSeR-class throughput requires fewer/shorter generations or concurrent batching

In [8]:
scenario = pd.DataFrame([
    {
        "scenario": "Observed C5",
        "evidence_class": "verified",
        "mean_s_per_item": observed_wall_mean,
        "full226_hours": C5_PUBLISHED_BATCH_WALL_SECONDS / 3600,
        "quality_risk": "none; observed result",
    },
    {
        "scenario": "Warm persistent resources only",
        "evidence_class": "measured components; estimated total",
        "mean_s_per_item": persistent_warm_mean,
        "full226_hours": persistent_warm_mean * 226 / 3600,
        "quality_risk": "low if object state is isolated and outputs replay-identical",
    },
    {
        "scenario": "Ideal 2-way question concurrency after warm-up",
        "evidence_class": "theoretical lower bound",
        "mean_s_per_item": persistent_warm_mean / 2,
        "full226_hours": persistent_warm_mean * 226 / 3600 / 2,
        "quality_risk": "requires deterministic concurrency and GPU contention test",
    },
    {
        "scenario": "Ideal 4-way question concurrency after warm-up",
        "evidence_class": "theoretical lower bound",
        "mean_s_per_item": persistent_warm_mean / 4,
        "full226_hours": persistent_warm_mean * 226 / 3600 / 4,
        "quality_risk": "requires continuous batching/backend validation",
    },
    {
        "scenario": "Observed ParSeR batch wall",
        "evidence_class": "verified",
        "mean_s_per_item": float(parser_run_summary["wall_seconds"]) / int(parser_agg["n_items"]),
        "full226_hours": float(parser_run_summary["wall_seconds"]) / 3600,
        "quality_risk": "different pipeline; no abstention/citation contract",
    },
])
display(scenario.round(3))

,scenario,evidence_class,mean_s_per_item,full226_hours,quality_risk
0,Observed C5,verified,138.731,8.709,none; observed result
1,Warm persistent resources only,measured components; estimated total,127.717,8.018,low if object state is isolated and outputs re...
2,Ideal 2-way question concurrency after warm-up,theoretical lower bound,63.859,4.009,requires deterministic concurrency and GPU con...
3,Ideal 4-way question concurrency after warm-up,theoretical lower bound,31.929,2.004,requires continuous batching/backend validation
4,Observed ParSeR batch wall,verified,35.064,2.201,different pipeline; no abstention/citation con...


### Validation retries consume a measurable but secondary share

In [9]:
retry_rows = attempts[attempts["attempt"] > 1]
retry_summary = pd.DataFrame([{
    "total_model_attempts": len(attempts),
    "logical_skill_invocations": int((attempts["attempt"] == 1).sum()),
    "retry_attempts": len(retry_rows),
    "invalid_or_truncated_attempts": int((attempts["outcome"] != "valid").sum()),
    "retry_generated_tokens": int(retry_rows["response_tokens"].sum()),
    "retry_interval_seconds": float(retry_rows["interval_s"].sum()),
    "retry_share_of_latency": float(retry_rows["interval_s"].sum() / c5_detail["latency_s"].sum()),
}])
display(retry_summary.round(3))
display(attempts.groupby(["skill_id", "outcome"]).size().rename("attempts").reset_index())

,total_model_attempts,logical_skill_invocations,retry_attempts,invalid_or_truncated_attempts,retry_generated_tokens,retry_interval_seconds,retry_share_of_latency
0,1126,1080,46,59,50600,1152.66,0.037


,skill_id,outcome,attempts
0,S1,valid,428
1,S1,validation_error,27
2,S2,truncated,1
3,S2,valid,426
4,S2,validation_error,10
5,S3,skill_error,18
6,S3,valid,213
7,S3,validation_error,3


## Takeaways

1. **C5 already fixed D4's worst runtime pathology:** its mean recorded latency is roughly 2.5× faster than D4, while precision, macro F1, answer rate, and execution stability improved.
2. **ParSeR still owns the throughput and answer-text frontier:** its full batch is roughly 4× faster per attempted item and its end-to-end Token-F1 is higher, but it lacks C5's ABSTAIN and deterministic citation-integrity contract.
3. **Warm-batch refactoring is necessary hygiene, not the overwhelming win:** measured repeated process/resource costs explain only about one-tenth of C5 wall time.
4. **The dominant lever is model-call topology:** each extra retrieval round adds a gap-planning S1 call and a coverage S2 call. Rich JSON generations make the average path substantially heavier than ParSeR's three compact stages.
5. **The credible Pareto path is staged:** first prove warm-path output equivalence, then test 2/4-way continuous batching, then reduce S1/S2 generation cost with compact schemas or a smaller specialist model while keeping the 27B model for final S3 and hard-case escalation.
6. **Quality gates must be joint:** no speed change should pass unless C5 macro F1 and failure rate remain non-inferior, D4's 2-hop complete-evidence gap improves, and citation integrity remains 100% on ANSWER outputs.

## 7. 추가 라운드의 gold 회수 가치

라운드를 줄여도 되는지 판단하려면 단순 상관관계가 아니라 실제 후보 집합이 언제 gold-complete가 되었는지를 확인해야 한다. 아래 셀은 공식 `evaluate_dev_runs.py`의 `load_dataset`, `build_gold_groups`, `precision_recall_f1`를 그대로 사용한다. 각 문항의 최종 `candidate_provisions`를 `retrieval_round <= r`로 누적하여 1·2·3라운드 시점의 recall-only complete를 재계산한다.

In [10]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.evaluate_dev_runs import (
    load_dataset,
    load_corpus,
    build_gold_groups,
    precision_recall_f1,
)

DATASET = ROOT / "data/koblex/qa/test-00000-of-00001.parquet"
CORPUS = ROOT / "data/koblex/normalized/statute.jsonl"
question_ids = c5_detail["question_id"].astype(str).tolist()
questions = load_dataset(DATASET)
wanted_indexes = {
    str(context["index"])
    for question_id in question_ids
    for context in questions[question_id]["contexts"]
}
gold_corpus = load_corpus(CORPUS, wanted_indexes)
groups_by_question = build_gold_groups(questions, question_ids, gold_corpus)

round_gold_rows = []
for row in c5_detail.itertuples(index=False):
    result = read_json(C5_RUN_ROOT / str(row.run_id) / "result.json")
    state = result["state"]
    candidates = state.get("candidate_provisions", [])
    metrics = {}
    for retrieval_round in (1, 2, 3):
        cumulative_ids = {
            str(candidate["provision_id"])
            for candidate in candidates
            if int(candidate.get("retrieval_round") or 0) <= retrieval_round
        }
        metrics[retrieval_round] = precision_recall_f1(
            cumulative_ids, groups_by_question[str(row.question_id)]
        )
    accepted = precision_recall_f1(
        state.get("accepted_provision_ids", []),
        groups_by_question[str(row.question_id)],
    )
    first_complete_round = next(
        (r for r in (1, 2, 3) if metrics[r]["complete"]), np.nan
    )
    round_gold_rows.append({
        "question_id": str(row.question_id),
        "n_hops": int(row.n_hops),
        "rounds_used": int(state["retrieval_rounds_used"]),
        "first_complete_round": first_complete_round,
        "accepted_complete": accepted["complete"],
        **{f"round_{r}_complete": metrics[r]["complete"] for r in (1, 2, 3)},
        **{f"round_{r}_recall": metrics[r]["recall"] for r in (1, 2, 3)},
    })
round_gold = pd.DataFrame(round_gold_rows)

round_gold_summary = pd.DataFrame([
    {
        "cumulative_round": r,
        "candidate_complete_items": int(round_gold[f"round_{r}_complete"].sum()),
        "candidate_complete_rate": round_gold[f"round_{r}_complete"].mean(),
        "candidate_macro_recall": round_gold[f"round_{r}_recall"].mean(),
        "new_complete_at_round": int((round_gold["first_complete_round"] == r).sum()),
    }
    for r in (1, 2, 3)
])
display(round_gold_summary.round(4))

continuation_value = pd.DataFrame([
    {
        "round_entered": r,
        "questions_entering": int((round_gold["rounds_used"] >= r).sum()),
        "already_gold_complete_before_entry": int(
            ((round_gold["rounds_used"] >= r) & round_gold[f"round_{r-1}_complete"]).sum()
        ),
        "newly_gold_complete_in_round": int(
            ((round_gold["rounds_used"] >= r)
             & ~round_gold[f"round_{r-1}_complete"]
             & round_gold[f"round_{r}_complete"]).sum()
        ),
        "still_incomplete_after_round": int(
            ((round_gold["rounds_used"] >= r) & ~round_gold[f"round_{r}_complete"]).sum()
        ),
    }
    for r in (2, 3)
])
display(continuation_value)

first_complete_by_hop = (
    round_gold.assign(
        first_complete_round=round_gold["first_complete_round"].fillna("never").astype(str)
    )
    .groupby(["n_hops", "first_complete_round"])
    .size()
    .rename("items")
    .reset_index()
)
display(first_complete_by_hop.pivot(index="n_hops", columns="first_complete_round", values="items").fillna(0).astype(int))

assert round_gold_summary["candidate_complete_items"].tolist() == [89, 122, 138]
assert continuation_value["newly_gold_complete_in_round"].tolist() == [33, 16]
assert int(round_gold["accepted_complete"].sum()) == 124


,cumulative_round,candidate_complete_items,candidate_complete_rate,candidate_macro_recall,new_complete_at_round
0,1,89,0.3938,0.5951,89
1,2,122,0.5398,0.7279,33
2,3,138,0.6106,0.7758,16


,round_entered,questions_entering,already_gold_complete_before_entry,newly_gold_complete_in_round,still_incomplete_after_round
0,2,128,15,33,80
1,3,76,22,16,38


first_complete_round,1.0,2.0,3.0,never
n_hops,,,,
1,46,5,1,5
2,41,23,13,50
3,2,5,2,33


### 해석

- 1라운드 후보만으로 complete인 문항은 89/226이다. 2라운드가 33건, 3라운드가 16건을 추가로 회수하여 최종 후보 complete는 138건이 된다.
- 따라서 2·3라운드를 일괄 제거하는 방식은 품질 비열등 조건을 통과할 수 없다. 특히 2-hop에서 새로 complete가 된 문항은 2라운드 23건, 3라운드 13건이다.
- 다만 2라운드 진입 128건 중 15건, 3라운드 진입 76건 중 22건은 evaluator gold 기준으로 이미 complete였다. 이는 oracle 진단이지 런타임이 직접 알 수 있는 신호는 아니므로, 실제 조기 종료에는 S2 coverage 판정의 calibration 검증이 필요하다.
- 후보 complete 138건에서 최종 accepted complete 124건으로 14건이 줄어든다. 속도와 품질을 함께 개선하려면 (a) D4→C5 분석에서 확인된 round-selected 병합 손실을 고쳐 gold를 더 일찍 넣고, (b) 이미 충분한 후보에 대한 추가 S1/S2 호출을 줄이며, (c) 뒤 라운드 S2 입력을 누적 전체가 아니라 신규·미해결 증거 중심으로 축약하는 것이 타당하다.